# 2. Baseline-модель — Cars Datasets 2025
**Цель:** построить простую baseline-модель для предсказания цены автомобиля, оценить её качество и зафиксировать метрики для сравнения с будущими экспериментами.

**Модель:** Ridge-регрессия на логарифме цены (из-за правосторонней асимметрии целевой переменной).

**Метрики:** MAE, RMSE (в USD), R².

In [3]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from src.preprocessing import load_data, clean_data, encode_categorical, split_data, scale_features

In [4]:
# Загрузка и очистка данных
df_raw = load_data('../data/raw/Cars_Datasets_2025.csv')
df_clean = clean_data(df_raw.copy())

print(f"Размер после очистки: {df_clean.shape}")
df_clean.head()

Загружено 1218 строк, 11 столбцов (кодировка: latin-1)
Колонки: ['Company Names', 'Cars Names', 'Engines', 'CC/Battery Capacity', 'HorsePower', 'Total Speed', 'Performance(0 - 100 )KM/H', 'Cars Prices', 'Fuel Types', 'Seats', 'Torque']
Удалено дубликатов: 4
Пропуски в колонках:
CC_Battery            206
HP                    185
Speed                   1
Acceleration_0_100     13
Price                   6
Seats                  12
Torque                 12
dtype: int64
Итоговый размер: 1214 строк, 10 столбцов
Размер после очистки: (1214, 10)


,Company,Engine,CC_Battery,HP,Speed,Acceleration_0_100,Price,Fuel,Seats,Torque
0,FERRARI,V8,3990.0,963.0,340.0,2.5,1100000.0,plug in hyrbrid,2,800.0
1,ROLLS ROYCE,V12,6749.0,563.0,250.0,5.3,460000.0,Petrol,5,900.0
2,Ford,1.2L Petrol,1200.0,77.5,165.0,10.5,13500.0,Petrol,5,120.0
3,MERCEDES,V8,3982.0,630.0,250.0,3.2,161000.0,Petrol,4,900.0
4,AUDI,V10,5204.0,602.0,320.0,3.6,253290.0,Petrol,2,560.0


In [5]:
# Создание логарифма цены (таргет)
if 'Price' in df_clean.columns:
    # Проверка, что все цены положительные
    assert (df_clean['Price'] > 0).all(), "Обнаружены неположительные цены"
    df_clean['Log_Price'] = np.log1p(df_clean['Price'])
    print("Создан Log_Price = log1p(Price)")
else:
    raise ValueError("Колонка Price отсутствует после очистки")

Создан Log_Price = log1p(Price)


In [6]:
# Подготовка признаков
df_features = df_clean.drop(columns=['Price'])

# Кодирование категориальных признаков
df_encoded = encode_categorical(df_features, target_col='Log_Price')

print(f"Фичей после кодирования: {df_encoded.shape[1]}")
df_encoded.head()

Фичей после кодирования: 420


,CC_Battery,HP,Speed,Acceleration_0_100,Seats,Torque,Log_Price,Company_AUDI,Company_Acura,Company_BENTLEY,...,Fuel_Petrol,Fuel_Petrol (Hybrid),"Fuel_Petrol, Diesel","Fuel_Petrol, Hybrid",Fuel_Petrol/AWD,Fuel_Petrol/Diesel,Fuel_Petrol/EV,Fuel_Petrol/Hybrid,Fuel_Plug-in Hybrid,Fuel_plug in hyrbrid
0,3990.0,963.0,340.0,2.5,2,800.0,13.910822,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,6749.0,563.0,250.0,5.3,5,900.0,13.038984,False,False,False,...,True,False,False,False,False,False,False,False,False,False
2,1200.0,77.5,165.0,10.5,5,120.0,9.510519,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,3982.0,630.0,250.0,3.2,4,900.0,11.989166,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,5204.0,602.0,320.0,3.6,2,560.0,12.442294,True,False,False,...,True,False,False,False,False,False,False,False,False,False


In [7]:
# Разделение на train/val/test (60/20/20)
X_train, X_val, X_test, y_train_log, y_val_log, y_test_log = split_data(
    df_encoded, target_col='Log_Price', test_size=0.2, val_size=0.2, random_state=RANDOM_STATE
)

Train: 728 samples
Val:   243 samples
Test:  243 samples


In [8]:
# Масштабирование признаков
X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_features(X_train, X_val, X_test)

print(f"Train scaled shape: {X_train_scaled.shape}")
print(f"Val scaled shape:   {X_val_scaled.shape}")
print(f"Test scaled shape:  {X_test_scaled.shape}")

Train scaled shape: (728, 419)
Val scaled shape:   (243, 419)
Test scaled shape:  (243, 419)


In [9]:
# Обучение Ridge-регрессии на логарифме цены
model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train_log)

print("Модель обучена")

Модель обучена


In [10]:
# Функция для вычисления метрик
def evaluate_model(model, X, y_log, y_orig, scaler):
    """
    Предсказание логарифма, обратное преобразование и расчёт метрик.
    y_orig – исходные цены в долларах.
    """
    pred_log = model.predict(X)
    # Обратное преобразование: expm1, так как использовали log1p
    pred = np.expm1(pred_log)
    mae = mean_absolute_error(y_orig, pred)
    rmse = np.sqrt(mean_squared_error(y_orig, pred))
    r2 = r2_score(y_orig, pred)
    return mae, rmse, r2, pred

# Оценим на train, val, test (y_orig = expm1(y_log))
y_train_orig = np.expm1(y_train_log)
y_val_orig = np.expm1(y_val_log)
y_test_orig = np.expm1(y_test_log)

train_mae, train_rmse, train_r2, train_pred = evaluate_model(model, X_train_scaled, y_train_log, y_train_orig, scaler)
val_mae, val_rmse, val_r2, val_pred = evaluate_model(model, X_val_scaled, y_val_log, y_val_orig, scaler)
test_mae, test_rmse, test_r2, test_pred = evaluate_model(model, X_test_scaled, y_test_log, y_test_orig, scaler)

In [11]:
# Вывод результатов
results = pd.DataFrame({
    'Dataset': ['Train', 'Validation', 'Test'],
    'MAE (USD)': [train_mae, val_mae, test_mae],
    'RMSE (USD)': [train_rmse, val_rmse, test_rmse],
    'R²': [train_r2, val_r2, test_r2]
})

print("\nBaseline (Ridge на log1p(Price)):")
print(results.to_string(index=False))

# Выделим итоговые тестовые метрики
print(f"\nИтоговый тест: MAE = {test_mae:,.0f} USD, RMSE = {test_rmse:,.0f} USD, R² = {test_r2:.3f}")


Baseline (Ridge на log1p(Price)):
   Dataset    MAE (USD)    RMSE (USD)       R²
     Train 54828.124739 563210.977182 0.585781
Validation 50117.079398 325153.162211 0.428418
      Test 31096.772070 130012.039801 0.625754

Итоговый тест: MAE = 31,097 USD, RMSE = 130,012 USD, R² = 0.626


In [ ]:

# Сохранение модели и скейлера для дальнейшего использования
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/baseline_ridge.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print("Модель и скейлер сохранены в папке models/")
print("  - models/baseline_ridge.pkl")
print("  - models/scaler.pkl")

## Выводы
- Baseline-модель на основе Ridge-регрессии и логарифма цены показала адекватные результаты.
- Метрики на тесте: MAE = 31 USD, RMSE = 130 USD, R² =0.626
- Модель будет служить точкой отсчёта для сравнения с более сложными алгоритмами (RandomForest, Gradient Boosting, ансамбли) и фича-инжинирингом.
- Сильное расхождение между train и val/test может указывать на переобучение, но для baseline это допустимо.